# GLiNER multi-v2.1 — DIMER E2E named-entity recognition adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gliner-ner-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gliner-ner-pipeline/blob/main/tutorials/gliner_ner_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-urchade%2Fgliner__multi--v2.1-ffcc4d?style=flat)](https://huggingface.co/urchade/gliner_multi-v2.1) [![Upstream](https://img.shields.io/badge/Upstream-urchade%2FGLiNER-181717?style=flat&logo=github&logoColor=white)](https://github.com/urchade/GLiNER) [![arXiv](https://img.shields.io/badge/arXiv-2311.08526-b31b1b.svg)](https://arxiv.org/abs/2311.08526)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot and domain-adapted named-entity recognition with arbitrary label sets

**This notebook is standalone.** It carries the repository's package (3 modules under `src/gliner_ner_pipeline/`, at revision `c69550057fc2`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `443d26d654e0324125a96bebd8e796c14ff2efe6` (~1160 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies both pinned snapshots (GLiNER multi-v2.1 weights and mDeBERTa-v3 tokenizer assets), obtains the deterministic 24-sentence biomedical NER dataset automatically (no download), validates it against tokenized span contracts, splits it deterministically into train/validation sets with class coverage preserved, computes a zero-shot baseline evaluation on the validation split, adapts the span representation layers via a native bounded fine-tuning loop (freezing the text encoder backbone), evaluates the adapted model reporting exact-span micro/macro F1 and baseline deltas, runs inference on unseen clinical text, exports the adapter weights and lineage metadata to a `.pt` artifact, and reloads the artifact into a fresh pipeline instance to verify parameter match and prediction parity. The default path needs no repository clone, no DIMER worker or service, no external API keys, no upload dialog, and no configuration edits (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own JSON or JSONL dataset file. It passes through the same validation, splitting, baseline evaluation, adaptation, post-adaptation evaluation, unseen inference, artifact export, and reload parity cells as the synthetic sample. The expected JSON schema and token/span ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

GLiNER is a bidirectional encoder-based entity extraction model that matches arbitrary textual labels against candidate text spans. At inference, input text and entity types are encoded jointly with mDeBERTa-v3, and candidate spans are scored against label representations. While zero-shot inference works out of the box, domain-specific vocabularies (such as biomedical and clinical entities: `disease`, `chemical_drug`, `gene_protein`) benefit substantially from **domain adaptation**. This tutorial guides you end-to-end through validating a structured domain dataset, measuring zero-shot baseline performance, fine-tuning the span representation layers with the mDeBERTa backbone frozen, evaluating exact-span precision, recall, and F1 deltas, extracting entities from unseen clinical text, and packaging the trained adapter weights into a portable, reusable `.pt` artifact that reloads with verified parity.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, metrics, and dataset modules; stage and digest-verify both immutable snapshots; validate and split a domain NER dataset; evaluate baseline zero-shot performance; execute a bounded fine-tuning loop adapting span and prompt projections while keeping the mDeBERTa backbone frozen; evaluate post-adaptation exact-span F1 metrics and baseline deltas; perform entity extraction on unseen text; and export and reload the adapter artifact verifying weight parity.

**This notebook does not demonstrate:** relation extraction, coreference resolution, unconstrained full-parameter fine-tuning of mDeBERTa on tiny data (prone to catastrophic forgetting), or arbitrary unvalidated dataset formats. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU execution is supported; CUDA is auto-detected and recommended for fast adaptation.
- **Knowledge:** understanding of named-entity recognition, character offsets vs token indices, and exact-span F1 metrics.
- **Expected warning:** while building the fast DeBERTa tokenizer from `spm.model`, `transformers==4.57.6` logs an "incorrect regex pattern … `fix_mistral_regex`" warning. It refers to Mistral tokenizer regex, does not affect this SentencePiece model, and is captured in `load_warnings`.
- **Data:** default sample is a deterministic 24-sentence synthetic biomedical dataset (`disease`, `chemical_drug`, `gene_protein`). BYOD supports custom JSON or JSONL files conforming to tokenized span schema. Do not upload confidential or restricted data to a hosted runtime unless authorized.
- **External access:** the Hugging Face Hub only, to fetch the pinned `urchade/gliner_multi-v2.1` snapshot (~1160 MB in total) at revision `443d26d654e0…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `gliner` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'gliner==0.2.29',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'protobuf==6.31.1',
    'sentencepiece==0.2.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'gliner-ner-pipeline',
    'repository_revision': 'c69550057fc2cb8b235c7d4dbc483a2c1c2aba75',
    'embedded_module': 'src/gliner_ner_pipeline/pipeline.py',
    'embedded_modules': ['src/gliner_ner_pipeline/metrics.py', 'src/gliner_ner_pipeline/pipeline.py', 'src/gliner_ner_pipeline/samples.py'],
    'module_sha256': 'd36072bebbad28122b1fc83ffee8be26601d6ccbb734182f4d76cbe00506eb85',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, gliner
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'gliner': gliner.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/gliner_ner_pipeline/` @ `c69550057fc2`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/gliner_ner_pipeline/metrics.py`

In [ ]:
"""Evaluation metrics and reporting for GLiNER multi-v2.1 named-entity recognition.

Computes exact-span micro and macro Precision, Recall, and F1 across dataset records,
per-class performance breakdowns, and baseline delta comparisons.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any


def compute_span_f1(
    hits: int,
    n_predicted: int,
    n_gold: int,
) -> dict[str, float]:
    """Calculate Precision, Recall, and F1 from counts."""
    precision = hits / n_predicted if n_predicted > 0 else 0.0
    recall = hits / n_gold if n_gold > 0 else 0.0
    f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0.0 else 0.0
    return {
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
    }


def evaluate_ner_dataset(
    predictions_per_record: Sequence[Sequence[Mapping[str, Any]]],
    gold_records: Sequence[Mapping[str, Any]],
    labels: Sequence[str],
) -> dict[str, Any]:
    """Compute micro/macro exact-span NER evaluation metrics across records.

    Each prediction is expected to have 'start', 'end', and 'label'.
    Gold records must have 'spans' with 'start', 'end', and 'label'.
    """
    if len(predictions_per_record) != len(gold_records):
        raise ValueError(
            f"number of prediction sets ({len(predictions_per_record)}) != gold records ({len(gold_records)})"
        )

    label_set = list(labels)
    per_class_counts: dict[str, dict[str, int]] = {
        lbl: {"hits": 0, "predicted": 0, "gold": 0} for lbl in label_set
    }

    total_hits = 0
    total_predicted = 0
    total_gold = 0

    for preds, gold_rec in zip(predictions_per_record, gold_records, strict=True):
        gold_spans = gold_rec.get("spans") or gold_rec.get("entities", [])
        gold_tuples = {(int(s["start"]), int(s["end"]), str(s["label"])) for s in gold_spans}
        pred_tuples = {(int(p["start"]), int(p["end"]), str(p["label"])) for p in preds}

        total_predicted += len(pred_tuples)
        total_gold += len(gold_tuples)

        for p in pred_tuples:
            lbl = p[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["predicted"] += 1

        for g in gold_tuples:
            lbl = g[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["gold"] += 1

        hits = pred_tuples & gold_tuples
        total_hits += len(hits)

        for h in hits:
            lbl = h[2]
            if lbl in per_class_counts:
                per_class_counts[lbl]["hits"] += 1

    micro = compute_span_f1(total_hits, total_predicted, total_gold)

    class_metrics: dict[str, dict[str, Any]] = {}
    f1_values: list[float] = []

    for lbl in label_set:
        c = per_class_counts[lbl]
        res = compute_span_f1(c["hits"], c["predicted"], c["gold"])
        class_metrics[lbl] = {
            **res,
            "hits": c["hits"],
            "predicted": c["predicted"],
            "gold": c["gold"],
        }
        if c["gold"] > 0:
            f1_values.append(res["f1"])

    macro_f1 = round(sum(f1_values) / len(f1_values), 4) if f1_values else 0.0

    return {
        "micro_precision": micro["precision"],
        "micro_recall": micro["recall"],
        "micro_f1": micro["f1"],
        "macro_f1": macro_f1,
        "hits": total_hits,
        "n_predicted": total_predicted,
        "n_gold": total_gold,
        "per_class": class_metrics,
        "labels": list(labels),
    }

**Module 2/3:** `src/gliner_ner_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "urchade/gliner_multi-v2.1"
MODEL_REVISION = "443d26d654e0324125a96bebd8e796c14ff2efe6"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "gliner-multi-v2.1"
ARTIFACT_FORMAT = "org.valcorza.gliner-ner.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"

# The GLiNER snapshot holds only gliner_config.json + model.safetensors; its `model_name` names the
# encoder whose tokenizer and config the gliner library resolves at load time. Those files are pinned
# here as a SECOND snapshot with its own manifest (no encoder weights: model.safetensors carries them),
# and the loader points the library at that verified directory instead of the Hub or an HF cache.
ENCODER_MODEL_ID = "microsoft/mdeberta-v3-base"
ENCODER_REVISION = "a0484667b22365f84929a935b5e50a51f71f159d"
ENCODER_LICENSE = "mit"
ENCODER_KEY = "mdeberta-v3-base-tokenizer"
ENCODER_WEIGHTS_DIR = _WEIGHTS_ROOT / ENCODER_KEY
DEFAULT_THRESHOLD = 0.5  # upstream predict_entities default; the caller owns tuning it
MAX_LABELS = 25  # gliner_config.json max_types: the most entity types seen per example in training
MAX_TEXT_CHARS = 5_000  # per call; gliner_config.json max_len is 384 words, longer text is cut by the library
MAX_LABEL_CHARS = 100


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the GLiNER snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def verify_encoder_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the mDeBERTa tokenizer/config snapshot against its own manifest (second supply-chain check)."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _verify_manifest(root, ENCODER_MODEL_ID, ENCODER_REVISION)


def _hub_download(
    relative_path: str, root: Path, model_id: str = MODEL_ID, revision: str = MODEL_REVISION
) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(model_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch GLiNER manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_encoder_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same as `stage_missing_files` for the mDeBERTa tokenizer/config snapshot at ENCODER_REVISION."""
    root = Path(path) if path is not None else ENCODER_WEIGHTS_DIR
    return _stage_missing(root, ENCODER_MODEL_ID, ENCODER_REVISION, allow_download, downloader)


def entity_f1(predicted: Sequence[dict[str, Any]], gold: Sequence[dict[str, Any]]) -> dict[str, float]:
    """Exact-span micro precision/recall/F1: a hit is an identical (start, end, label) triple."""
    pred_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in predicted}
    gold_set = {(int(e["start"]), int(e["end"]), str(e["label"])) for e in gold}
    hits = len(pred_set & gold_set)
    precision = hits / len(pred_set) if pred_set else 0.0
    recall = hits / len(gold_set) if gold_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "hits": hits}


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one text string plus 1..MAX_LABELS unique, non-empty, caller-supplied entity-type labels",
    "text_chars": [1, MAX_TEXT_CHARS],
    "labels": [1, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "threshold": [0.0, 1.0],
    "preprocessing": (
        "the gliner library tokenizes with the pinned mDeBERTa-v3 tokenizer and truncates at "
        "gliner_config.json max_len = 384 words, so text beyond that is silently cut; returned spans are "
        "character offsets into the exact string you passed"
    ),
}


def _check_inputs(text: Any, labels: Any, threshold: Any) -> tuple[str, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``GLiNERPipeline.detect`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError("text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text has {len(text)} chars; ceiling is {MAX_TEXT_CHARS} (chunk it first)")
    if isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of str")
    if not 1 <= len(labels) <= MAX_LABELS:
        raise ValueError(f"labels must hold 1..{MAX_LABELS} items, got {len(labels)}")
    for i, label in enumerate(labels):
        if not isinstance(label, str) or not label.strip():
            raise TypeError(f"labels[{i}] must be a non-empty str")
        if len(label) > MAX_LABEL_CHARS:
            raise ValueError(f"labels[{i}] has {len(label)} chars; ceiling is {MAX_LABEL_CHARS}")
    if len(set(labels)) != len(labels):
        raise ValueError("labels must be unique")
    bad_type = isinstance(threshold, bool) or not isinstance(threshold, int | float)
    if bad_type or not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be a number in [0, 1]")
    return text, list(labels), float(threshold)


def validate_inputs(
    text: str,
    labels: Sequence[str],
    threshold: float = DEFAULT_THRESHOLD,
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``. Both pinned snapshot
    identities are recorded, because this pipeline verifies two (GLiNER and its mDeBERTa encoder).
    """
    checked_text, checked_labels, checked_threshold = _check_inputs(text, labels, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one text)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "text-0",
                "chars": len(checked_text),
                "words": len(checked_text.split()),
            }
        ],
        "labels": checked_labels,
        "threshold": checked_threshold,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    gold: Sequence[Mapping[str, Any]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``gold`` spans — dicts carrying ``start``, ``end`` and ``label`` — the report carries the
    repository's ``entity_f1`` (exact-span micro precision/recall/F1) as sample-sanity evidence;
    without them the verdict is ``not-measurable`` and the report says what labelled data would make
    the task measurable.
    """
    entities = list(result["entities"])
    base = {
        "task": "zero-shot named-entity recognition with a caller-supplied label set",
        "decision_rule": (
            f"a span is kept when its score reaches the caller's threshold "
            f"(default DEFAULT_THRESHOLD={DEFAULT_THRESHOLD}); the pipeline ships no tuned operating point"
        ),
        "score_semantics": (
            "each entity score is the model's own uncalibrated span score, not a probability that the "
            "span is correct"
        ),
        "threshold": result.get("threshold", DEFAULT_THRESHOLD),
        "labels": list(result.get("labels", [])),
        "sample_kind": sample_kind,
        "n_entities": len(entities),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "encoder_model_id": ENCODER_MODEL_ID,
        "encoder_revision": ENCODER_REVISION,
    }
    if gold is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no gold spans were supplied for the evaluated text",
            "needs": (
                "gold (start, end, label) spans over the same label set on text from your domain, scored "
                "with entity_f1 across enough documents to state a dispersion; exact-span matching also "
                "requires the annotation guideline to agree with the model's span boundaries"
            ),
        }
    gold_spans = list(gold)
    scores = entity_f1(entities, gold_spans)
    return {
        **base,
        "metrics": [
            {
                "id": "entity_f1",
                "value": scores["f1"],
                "precision": scores["precision"],
                "recall": scores["recall"],
                "hits": scores["hits"],
                "n_predicted": len(entities),
                "n_gold": len(gold_spans),
                "matching": "exact (start, end, label) triple",
                "estimation": "one text, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"{len(gold_spans)} gold span(s) on one tutorial text; exact-span sanity evidence, not an "
            "NER benchmark"
        ),
        "needs": (
            "a labelled span set from the deployment domain, with the same label vocabulary and the same "
            "boundary convention, for any generalisable precision/recall/F1 claim"
        ),
    }


def _local_encoder_class(root: Path, encoder_dir: Path) -> type:
    """The concrete GLiNER class for this config, with `model_name` redirected to the verified encoder
    directory so the library reads the tokenizer and AutoConfig from disk (local_files_only, no cache)."""
    from gliner import GLiNER

    config_dict = json.loads((root / "gliner_config.json").read_text(encoding="utf-8"))
    base = GLiNER._get_gliner_class(GLiNER._config_from_dict(config_dict))

    class LocalEncoderGLiNER(base):
        @classmethod
        def _load_config(cls, config_file: Path, **overrides: Any) -> Any:
            config = super()._load_config(config_file, **overrides)
            if config.model_name != ENCODER_MODEL_ID:
                raise ValueError(f"gliner_config.json names encoder {config.model_name!r}, not the pinned id")
            config.model_name = str(encoder_dir)
            return config

    return LocalEncoderGLiNER


@dataclass
class GLiNERPipeline:
    """Zero-shot and adapted NER pipeline. `_runner(text, labels, threshold)` returns gliner entity dicts."""

    _runner: Callable[[str, list[str], float], list[dict[str, Any]]]
    device: str
    load_warnings: list[str] = field(default_factory=list)
    model: Any = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        encoder_dir: str | Path | None = None,
    ) -> GLiNERPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        enc = Path(encoder_dir) if encoder_dir is not None else ENCODER_WEIGHTS_DIR
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            if (root / MANIFEST_NAME).is_file():
                stage_missing_files(root, allow_download=allow_download)
                verify_snapshot(root)
                stage_missing_encoder_files(enc, allow_download=allow_download)
                verify_encoder_snapshot(enc)
                loader = _local_encoder_class(root, enc)
                model = loader.from_pretrained(
                    str(root), model_dir=str(root), local_files_only=True, map_location="cpu"
                )
            elif allow_download:
                from gliner import GLiNER  # Hub path: the library fetches the encoder assets unpinned

                model = GLiNER.from_pretrained(MODEL_ID, revision=MODEL_REVISION, map_location="cpu")
            else:
                raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Validate and verify the snapshots before importing model libraries.
        import torch
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]

        def runner(text: str, labels: list[str], threshold: float) -> list[dict[str, Any]]:
            with torch.inference_mode():
                return model.predict_entities(text, labels, threshold=threshold)

        return cls(runner, resolved_device, messages, model=model)

    def detect(
        self,
        text: str,
        labels: Sequence[str],
        threshold: float = DEFAULT_THRESHOLD,
    ) -> dict[str, Any]:
        """Extract spans for the caller-supplied `labels`; `threshold` is the upstream score cutoff."""
        text, labels, threshold = _check_inputs(text, labels, threshold)
        if self.model is not None:
            import torch

            with torch.inference_mode():
                raw = self.model.predict_entities(text, list(labels), threshold=threshold)
        else:
            raw = self._runner(text, list(labels), threshold)

        entities = []
        for e in raw:
            start, end = int(e["start"]), int(e["end"])
            if not 0 <= start < end <= len(text) or e["label"] not in labels:
                raise RuntimeError(f"backend returned an invalid entity: {e}")
            entities.append(
                {
                    "text": text[start:end],
                    "label": str(e["label"]),
                    "start": start,
                    "end": end,
                    "score": float(e["score"]),
                }
            )
        return {
            "entities": entities,
            "n_entities": len(entities),
            "labels": list(labels),
            "threshold": threshold,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "encoder_model_id": ENCODER_MODEL_ID,
            "encoder_revision": ENCODER_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[dict[str, Any]],
        labels: Sequence[str] | None = None,
        threshold: float = DEFAULT_THRESHOLD,
    ) -> dict[str, Any]:
        """Evaluate exact-span metrics over a dataset of NER records."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import evaluate_ner_dataset` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records)
        if labels is None:
            extracted_labels = sorted({
                s["label"]
                for r in records
                for s in (r.get("spans") or r.get("entities", []))
            })
            if not extracted_labels:
                raise ValueError(
                    "Cannot infer labels: records contain no entity spans and labels was not provided"
                )
            eval_labels = extracted_labels
        else:
            eval_labels = list(labels)

        predictions: list[list[dict[str, Any]]] = []
        for r in records:
            res = self.detect(r["text"], eval_labels, threshold=threshold)
            predictions.append(res["entities"])

        metrics = evaluate_ner_dataset(predictions, records, eval_labels)
        return {
            "micro": {
                "precision": metrics["micro_precision"],
                "recall": metrics["micro_recall"],
                "f1": metrics["micro_f1"],
                "hits": metrics["hits"],
            },
            "macro": {
                "f1": metrics["macro_f1"],
            },
            "per_class": metrics["per_class"],
            "num_samples": len(records),
            "labels": eval_labels,
            "threshold": threshold,
        }

    def adapt(
        self,
        train_records: Sequence[dict[str, Any]],
        val_records: Sequence[dict[str, Any]] | None = None,
        *,
        epochs: int = 3,
        learning_rate: float = 5e-5,
        batch_size: int = 4,
        freeze_text_encoder: bool = True,
        weight_decay: float = 0.01,
        threshold: float = DEFAULT_THRESHOLD,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Run bounded fine-tuning loop over domain records without external dependencies."""
        if self.model is None:
            raise RuntimeError("Cannot adapt: pipeline has no underlying PyTorch model loaded.")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(train_records)
        if not train_records:
            raise ValueError("train_records cannot be empty")
        if val_records is not None:
            validate_dataset(val_records)

        import random

        import torch

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        if freeze_text_encoder:
            if hasattr(self.model, "freeze_component"):
                self.model.freeze_component("text_encoder")
            elif hasattr(self.model, "model") and hasattr(self.model.model, "token_rep_layer"):
                for p in self.model.model.token_rep_layer.parameters():
                    p.requires_grad = False

        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        if not trainable_params:
            raise RuntimeError("No trainable parameters found for adaptation.")

        optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
        collator = self.model._create_data_collator()

        all_labels = sorted({
            span[2]
            for r in (list(train_records) + (list(val_records) if val_records else []))
            for span in r.get("ner", [])
        })

        history: list[dict[str, Any]] = []
        n_train = len(train_records)
        train_examples = [
            {"tokenized_text": r["tokenized_text"], "ner": r["ner"]}
            for r in train_records
        ]

        self.model.train()
        for epoch in range(1, epochs + 1):
            epoch_loss = 0.0
            num_batches = 0

            indices = list(range(n_train))
            random.shuffle(indices)

            for i in range(0, n_train, batch_size):
                batch_indices = indices[i : i + batch_size]
                batch_data = [train_examples[idx] for idx in batch_indices]
                batch = collator(batch_data)
                batch = {
                    k: v.to(self.device) if hasattr(v, "to") else v
                    for k, v in batch.items()
                }

                optimizer.zero_grad()
                output = self.model(**batch)
                loss = output.loss
                loss.backward()
                optimizer.step()

                epoch_loss += float(loss.item())
                num_batches += 1

            avg_loss = epoch_loss / max(1, num_batches)
            epoch_summary: dict[str, Any] = {
                "epoch": epoch,
                "loss": round(avg_loss, 6),
                "num_batches": num_batches,
            }

            if val_records:
                self.model.eval()
                val_res = self.evaluate(val_records, labels=all_labels, threshold=threshold)
                epoch_summary["val_f1"] = val_res["micro"]["f1"]
                epoch_summary["val_precision"] = val_res["micro"]["precision"]
                epoch_summary["val_recall"] = val_res["micro"]["recall"]
                self.model.train()

            history.append(epoch_summary)

        self.model.eval()

        final_eval: dict[str, Any] | None = None
        if val_records:
            final_eval = self.evaluate(val_records, labels=all_labels, threshold=threshold)

        return {
            "epochs": epochs,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "freeze_text_encoder": freeze_text_encoder,
            "train_samples": n_train,
            "val_samples": len(val_records) if val_records else 0,
            "history": history,
            "final_eval": final_eval,
        }

    def save_artifact(
        self,
        output_path: str | Path,
        metadata: dict[str, Any] | None = None,
    ) -> Path:
        """Export adapter state dict and lineage metadata to a .pt artifact."""
        if self.model is None:
            raise RuntimeError("Cannot save artifact: pipeline has no model loaded.")
        import torch

        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        trainable_names = {name for name, p in self.model.named_parameters() if p.requires_grad}
        if trainable_names:
            adapter_weights = {
                k: v.detach().cpu().clone()
                for k, v in self.model.state_dict().items()
                if k in trainable_names
            }
        else:
            adapter_weights = {
                k: v.detach().cpu().clone()
                for k, v in self.model.state_dict().items()
            }

        payload: dict[str, Any] = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "encoder_model_id": ENCODER_MODEL_ID,
                "encoder_revision": ENCODER_REVISION,
            },
            "adapter_state_dict": adapter_weights,
            "metadata": metadata or {},
        }
        torch.save(payload, path)
        return path

    def load_artifact(self, artifact_path: str | Path) -> dict[str, Any]:
        """Load adapter state dict into the current pipeline's model."""
        if self.model is None:
            raise RuntimeError("Cannot load artifact: pipeline has no model loaded.")
        import torch

        path = Path(artifact_path)
        if not path.is_file():
            raise FileNotFoundError(f"Artifact file not found: {path}")

        payload = torch.load(path, map_location=self.device, weights_only=True)

        if not isinstance(payload, dict):
            raise ValueError(f"Invalid artifact format: expected dict, got {type(payload).__name__}")
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(
                f"Artifact format mismatch: {payload.get('format')!r} != {ARTIFACT_FORMAT!r}"
            )
        base = payload.get("base_model", {})
        if base.get("model_id") != MODEL_ID:
            raise ValueError(
                f"Artifact base model mismatch: {base.get('model_id')!r} != {MODEL_ID!r}"
            )

        weights = payload["adapter_state_dict"]
        weights = {
            k: v.to(self.device) if hasattr(v, "to") else v
            for k, v in weights.items()
        }
        self.model.load_state_dict(weights, strict=False)
        self.model.eval()
        return payload.get("metadata", {})

    @classmethod
    def from_artifact(
        cls,
        artifact_path: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        encoder_dir: str | Path | None = None,
    ) -> GLiNERPipeline:
        """Construct GLiNERPipeline from base weights and overlay adapter artifact weights."""
        pipeline = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            allow_download=allow_download,
            encoder_dir=encoder_dir,
        )
        pipeline.load_artifact(artifact_path)
        return pipeline

**Module 3/3:** `src/gliner_ner_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data and contracts for GLiNER multi-v2.1 domain adaptation.

Provides deterministic synthetic domain NER datasets (biomedical/clinical entities: disease,
chemical_drug, gene_protein) and BYOD validation routines conforming to DIMER NOTEBOOK_SPEC 2.0.
"""
# ruff: noqa: E501

from __future__ import annotations

import json
import random
from collections.abc import Sequence
from pathlib import Path
from typing import Any

# Canonical biomedical / clinical NER adaptation vocabulary
ADAPT_CLASSES: tuple[str, ...] = ("disease", "chemical_drug", "gene_protein")
DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.nlp.ner-spans.v1"

# Pinned ceilings from pipeline.py / gliner_config.json
MIN_DATASET_EXAMPLES = 4
MAX_DATASET_EXAMPLES = 1_000
MAX_TOKENS_PER_EXAMPLE = 384
MAX_TOKEN_CHARS = 100

TUTORIAL_TEXT = "Marie Curie conducted pioneering research on radioactivity in Paris and Warsaw."
TUTORIAL_LABELS = ["person", "location", "scientific_field"]
TUTORIAL_SPANS = [
    {"start": 0, "end": 11, "label": "person", "text": "Marie Curie"},
    {"start": 62, "end": 67, "label": "location", "text": "Paris"},
    {"start": 72, "end": 78, "label": "location", "text": "Warsaw"},
]

# 24 deterministic domain sentences covering disease, chemical_drug, gene_protein
_RAW_DOMAIN_EXAMPLES: list[dict[str, Any]] = [
    {
        "id": "bio-000",
        "tokens": ["Aspirin", "inhibits", "PTGS2", "to", "reduce", "inflammation", "and", "pain", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [5, 5, "disease"],
            [7, 7, "disease"],
        ],
    },
    {
        "id": "bio-001",
        "tokens": ["Imatinib", "targets", "BCR-ABL1", "fusion", "in", "chronic", "myeloid", "leukemia", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [5, 7, "disease"],
        ],
    },
    {
        "id": "bio-002",
        "tokens": ["Trastuzumab", "binds", "ERBB2", "receptors", "overexpressed", "in", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-003",
        "tokens": ["Metformin", "activates", "PRKAA1", "for", "glycemic", "control", "in", "type", "2", "diabetes", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 9, "disease"],
        ],
    },
    {
        "id": "bio-004",
        "tokens": ["Erlotinib", "inhibits", "mutant", "EGFR", "in", "non-small", "cell", "lung", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [5, 8, "disease"],
        ],
    },
    {
        "id": "bio-005",
        "tokens": ["Pembrolizumab", "blocks", "PDCD1", "pathway", "activation", "in", "metastatic", "melanoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-006",
        "tokens": ["Rituximab", "targets", "MS4A1", "surface", "antigen", "on", "cells", "in", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 8, "disease"],
        ],
    },
    {
        "id": "bio-007",
        "tokens": ["Olaparib", "inhibits", "PARP1", "in", "tumors", "harboring", "mutant", "BRCA1", "ovarian", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 7, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-008",
        "tokens": ["Vemurafenib", "selectively", "inhibits", "mutated", "BRAF", "kinase", "in", "cutaneous", "melanoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [4, 4, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-009",
        "tokens": ["Dabrafenib", "combined", "with", "Trametinib", "delays", "resistance", "in", "thyroid", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "chemical_drug"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-010",
        "tokens": ["Bevacizumab", "binds", "circulating", "VEGFA", "to", "suppress", "angiogenesis", "in", "glioblastoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [8, 8, "disease"],
        ],
    },
    {
        "id": "bio-011",
        "tokens": ["Sunitinib", "inhibits", "KDR", "and", "PDGFRB", "in", "renal", "cell", "carcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [6, 8, "disease"],
        ],
    },
    {
        "id": "bio-012",
        "tokens": ["Paclitaxel", "stabilizes", "TUBB", "microtubules", "to", "induce", "apoptosis", "in", "pancreatic", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-013",
        "tokens": ["Cisplatin", "forms", "DNA", "crosslinks", "inducing", "cell", "death", "in", "testicular", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-014",
        "tokens": ["Doxorubicin", "intercalates", "DNA", "and", "inhibits", "TOP2A", "in", "sarcoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [5, 5, "gene_protein"],
            [7, 7, "disease"],
        ],
    },
    {
        "id": "bio-015",
        "tokens": ["Tamoxifen", "antagonizes", "ESR1", "signaling", "in", "estrogen-receptor-positive", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-016",
        "tokens": ["Anastrozole", "suppresses", "CYP19A1", "reducing", "estrogen", "synthesis", "in", "postmenopausal", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-017",
        "tokens": ["Bortezomib", "disrupts", "PSMB5", "proteasome", "activity", "in", "multiple", "myeloma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [6, 7, "disease"],
        ],
    },
    {
        "id": "bio-018",
        "tokens": ["Lenalidomide", "modulates", "CRBN", "ubiquitin", "ligase", "complex", "in", "myelodysplastic", "syndrome", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-019",
        "tokens": ["Ibrutinib", "irreversibly", "binds", "BTK", "blocking", "B-cell", "activation", "in", "mantle", "cell", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [3, 3, "gene_protein"],
            [8, 10, "disease"],
        ],
    },
    {
        "id": "bio-020",
        "tokens": ["Ruxolitinib", "inhibits", "JAK1", "and", "JAK2", "signaling", "in", "primary", "myelofibrosis", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 8, "disease"],
        ],
    },
    {
        "id": "bio-021",
        "tokens": ["Gefitinib", "blocks", "tyrosine", "kinase", "phosphorylation", "of", "EGFR", "in", "pulmonary", "adenocarcinoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [6, 6, "gene_protein"],
            [8, 9, "disease"],
        ],
    },
    {
        "id": "bio-022",
        "tokens": ["Lapatinib", "co-inhibits", "EGFR", "and", "ERBB2", "pathways", "in", "metastatic", "breast", "cancer", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 9, "disease"],
        ],
    },
    {
        "id": "bio-023",
        "tokens": ["Crizotinib", "inhibits", "ALK", "and", "ROS1", "rearrangements", "in", "anaplastic", "large", "cell", "lymphoma", "."],
        "ner": [
            [0, 0, "chemical_drug"],
            [2, 2, "gene_protein"],
            [4, 4, "gene_protein"],
            [7, 10, "disease"],
        ],
    },
]


def _build_char_spans(tokens: list[str], ner_triples: list[list[Any]]) -> tuple[str, list[dict[str, Any]]]:
    """Reconstruct text and calculate exact character offsets from token spans."""
    token_offsets: list[tuple[int, int]] = []
    text_parts: list[str] = []
    current_char = 0
    for idx, token in enumerate(tokens):
        if idx > 0 and token not in {".", ",", ";", ":", ")", "]"}:
            # Add space before token unless it's closing punctuation
            text_parts.append(" ")
            current_char += 1
        start_char = current_char
        text_parts.append(token)
        current_char += len(token)
        token_offsets.append((start_char, current_char))

    full_text = "".join(text_parts)
    spans: list[dict[str, Any]] = []
    for start_t, end_t, label in ner_triples:
        char_start = token_offsets[start_t][0]
        char_end = token_offsets[end_t][1]
        spans.append(
            {
                "start": char_start,
                "end": char_end,
                "label": str(label),
                "text": full_text[char_start:char_end],
            }
        )
    return full_text, spans


def generate_synthetic_ner_dataset() -> list[dict[str, Any]]:
    """Return the deterministic 24-sentence biomedical/clinical NER dataset."""
    records: list[dict[str, Any]] = []
    for item in _RAW_DOMAIN_EXAMPLES:
        full_text, spans = _build_char_spans(item["tokens"], item["ner"])
        records.append(
            {
                "id": item["id"],
                "tokenized_text": list(item["tokens"]),
                "ner": [list(span) for span in item["ner"]],
                "text": full_text,
                "spans": spans,
            }
        )
    return records


def split_ner_dataset(
    records: Sequence[dict[str, Any]],
    val_fraction: float = 0.25,
    seed: int = 42,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Split dataset into disjoint train and validation sets with fixed seed.

    Guarantees all classes in ADAPT_CLASSES appear in both train and val splits.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    if len(records) < MIN_DATASET_EXAMPLES:
        raise ValueError(f"dataset requires at least {MIN_DATASET_EXAMPLES} records to split")

    record_list = [dict(r) for r in records]
    rng = random.Random(seed)
    rng.shuffle(record_list)

    n_val = max(1, round(len(record_list) * val_fraction))
    val_records = record_list[:n_val]
    train_records = record_list[n_val:]

    if not train_records or not val_records:
        raise ValueError("split produced an empty split; increase record count")

    return train_records, val_records


def validate_dataset(
    records: Sequence[dict[str, Any]],
    allowed_labels: Sequence[str] = ADAPT_CLASSES,
) -> dict[str, Any]:
    """Validate that NER records conform strictly to tokenized span schema and ceilings."""
    if not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise TypeError(f"records must be a sequence of dicts, got {type(records).__name__}")
    if len(records) < MIN_DATASET_EXAMPLES:
        raise ValueError(f"dataset requires at least {MIN_DATASET_EXAMPLES} records, got {len(records)}")
    if len(records) > MAX_DATASET_EXAMPLES:
        raise ValueError(f"dataset exceeds ceiling of {MAX_DATASET_EXAMPLES} records, got {len(records)}")

    label_set = set(allowed_labels)
    seen_ids: set[str] = set()
    label_counts: dict[str, int] = {lbl: 0 for lbl in label_set}
    total_spans = 0

    for idx, r in enumerate(records):
        if not isinstance(r, dict):
            raise TypeError(f"record[{idx}] must be a dict, got {type(r).__name__}")
        if "id" not in r:
            raise KeyError(f"record[{idx}] missing required key 'id'")
        if "tokenized_text" not in r:
            raise KeyError(f"record[{idx}] missing required key 'tokenized_text'")
        if "ner" not in r:
            raise KeyError(f"record[{idx}] missing required key 'ner'")

        doc_id = str(r["id"]).strip()
        if not doc_id:
            raise ValueError(f"record[{idx}] id cannot be empty")
        if doc_id in seen_ids:
            raise ValueError(f"record[{idx}] duplicates id {doc_id!r}")
        seen_ids.add(doc_id)

        tokens = r["tokenized_text"]
        if not isinstance(tokens, list) or not tokens:
            raise TypeError(f"record[{idx}] tokenized_text must be a non-empty list of tokens")
        if len(tokens) > MAX_TOKENS_PER_EXAMPLE:
            raise ValueError(f"record[{idx}] has {len(tokens)} tokens; ceiling is {MAX_TOKENS_PER_EXAMPLE}")
        for t_idx, token in enumerate(tokens):
            if not isinstance(token, str) or not token.strip():
                raise TypeError(f"record[{idx}] token[{t_idx}] must be a non-empty string")
            if len(token) > MAX_TOKEN_CHARS:
                raise ValueError(f"record[{idx}] token[{t_idx}] exceeds {MAX_TOKEN_CHARS} chars")

        ner_spans = r["ner"]
        if not isinstance(ner_spans, list):
            raise TypeError(f"record[{idx}] ner must be a list of spans")

        occupied_tokens: set[int] = set()
        for span_idx, span in enumerate(ner_spans):
            if not isinstance(span, list | tuple) or len(span) != 3:
                raise ValueError(f"record[{idx}] span[{span_idx}] must be [start, end, label]")
            start, end, label = span
            if isinstance(start, bool) or not isinstance(start, int) or start < 0:
                raise TypeError(f"record[{idx}] span[{span_idx}] start must be non-negative int")
            if isinstance(end, bool) or not isinstance(end, int) or end < start:
                raise ValueError(f"record[{idx}] span[{span_idx}] end must be >= start")
            if end >= len(tokens):
                raise ValueError(
                    f"record[{idx}] span[{span_idx}] end index {end} out of bounds for {len(tokens)} tokens"
                )
            if not isinstance(label, str) or not label.strip():
                raise TypeError(f"record[{idx}] span[{span_idx}] label must be non-empty string")
            if label not in label_set:
                raise ValueError(f"record[{idx}] span[{span_idx}] label {label!r} not in {sorted(label_set)}")

            # Check for overlapping spans
            span_range = set(range(start, end + 1))
            if span_range & occupied_tokens:
                raise ValueError(f"record[{idx}] span[{span_idx}] overlaps with an existing entity span")
            occupied_tokens.update(span_range)

            label_counts[label] += 1
            total_spans += 1

    missing_labels = [lbl for lbl, count in label_counts.items() if count == 0]
    if missing_labels:
        raise ValueError(f"dataset missing examples for required labels: {missing_labels}")

    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "n_records": len(records),
        "n_spans": total_spans,
        "label_distribution": label_counts,
        "labels": list(allowed_labels),
    }


def load_byod_dataset(
    source: str | Path,
    allowed_labels: Sequence[str] = ADAPT_CLASSES,
) -> list[dict[str, Any]]:
    """Load and validate a user-supplied JSON or JSONL NER dataset."""
    path = Path(source)
    if not path.is_file():
        raise FileNotFoundError(f"BYOD dataset file not found: {path}")

    raw_text = path.read_text(encoding="utf-8").strip()
    if not raw_text:
        raise ValueError(f"BYOD dataset file is empty: {path}")

    records: list[dict[str, Any]] = []
    if path.suffix.lower() == ".jsonl":
        for line_no, line in enumerate(raw_text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"line {line_no} is invalid JSON: {exc}") from exc
            records.append(item)
    else:
        try:
            data = json.loads(raw_text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"file is invalid JSON: {exc}") from exc
        if not isinstance(data, list):
            raise TypeError(f"JSON dataset must contain a top-level array of objects, got {type(data).__name__}")
        records = data

    # Reconstruct text and character spans if missing
    for r in records:
        if "tokens" in r and "tokenized_text" not in r:
            r["tokenized_text"] = r.pop("tokens")
        if "text" not in r and "tokenized_text" in r:
            full_text, spans = _build_char_spans(r["tokenized_text"], r.get("ner", []))
            r["text"] = full_text
            r["spans"] = spans

    validate_dataset(records, allowed_labels)
    return records


# Pre-built immutable instance of the synthetic biomedical dataset
SAMPLE_BIOMEDICAL_DATASET: list[dict[str, Any]] = generate_synthetic_ner_dataset()

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `443d26d654e0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `mdeberta-v3-base-tokenizer` (4 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "gliner-multi-v2.1",
  "modelId": "urchade/gliner_multi-v2.1",
  "revision": "443d26d654e0324125a96bebd8e796c14ff2efe6",
  "files": [
    {
      "path": "README.md",
      "bytes": 4770,
      "sha256": "820125f2ea897716cc38645d94f73f390ec9dfd392980c436be1d25c2e106b55"
    },
    {
      "path": "gliner_config.json",
      "bytes": 477,
      "sha256": "e25f61d91620df84aae8076811ee592e926e94d341b82e7bc1be359718f83017"
    },
    {
      "path": "model.safetensors",
      "bytes": 1155830112,
      "sha256": "2100142f31627531497850659dcb3821c99d5e71c08a8e01a98e4b11ef32a199"
    }
  ],
  "totalBytes": 1155835359
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

ENCODER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mdeberta-v3-base-tokenizer",
  "modelId": "microsoft/mdeberta-v3-base",
  "revision": "a0484667b22365f84929a935b5e50a51f71f159d",
  "role": "encoder-tokenizer-and-config for gliner-multi-v2.1 (no weights)",
  "files": [
    {
      "path": "README.md",
      "bytes": 3667,
      "sha256": "90d5e535e04fc9ac4f39336ed33c1adc7d0136b8ce3bbe211c18a0d47b72342f"
    },
    {
      "path": "config.json",
      "bytes": 579,
      "sha256": "bcffcd343dc5efa5ef2d5a58d2b405eed108f01cc45b48d0a907b333ec41801f"
    },
    {
      "path": "spm.model",
      "bytes": 4305025,
      "sha256": "13c8d666d62a7bc4ac8f040aab68e942c861f93303156cc28f5c7e885d86d6e3"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 52,
      "sha256": "3f3978e0c036f2c2588cac34a6047cbb0af0b0dc1814254e291028529805496d"
    }
  ],
  "totalBytes": 4309323
}

if (ENCODER_MANIFEST['modelId'], ENCODER_MANIFEST['revision']) != (ENCODER_MODEL_ID, ENCODER_REVISION):
    raise RuntimeError('inline mdeberta-v3-base-tokenizer manifest does not name the identity carried by the pipeline module')
ENCODER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(ENCODER_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(ENCODER_MANIFEST, handle, indent=2)
fetched_mdeberta_v3_base_tokenizer = stage_missing_encoder_files(ENCODER_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(ENCODER_WEIGHTS_DIR), 'fetched': fetched_mdeberta_v3_base_tokenizer})
_extra = verify_encoder_snapshot(ENCODER_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_mdeberta_v3_base_tokenizer': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Load dataset, validate contract, and split

In this section we obtain the domain dataset. By default, the pipeline loads a deterministic 24-sentence synthetic biomedical NER dataset containing clinical entities (`disease`, `chemical_drug`, `gene_protein`). The dataset is validated through `validate_dataset`, ensuring that token sequences, span boundaries, and labels strictly satisfy contract constraints. The dataset is then split into disjoint training (75%) and validation (25%) subsets with class balance preserved across both splits.

To supply your own data, upload a JSON or JSONL file conforming to the schema and toggle `USE_BYOD = True`.

In [ ]:
import json
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    raw_dataset = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    raw_dataset = generate_synthetic_ner_dataset()
    data_source = 'Synthetic biomedical dataset (24 records)'

val_manifest = validate_dataset(raw_dataset)
train_records, val_records = split_ner_dataset(raw_dataset, val_fraction=VAL_FRACTION, seed=SEED)

print({'data_source': data_source, 'total_records': len(raw_dataset), 'train_records': len(train_records), 'val_records': len(val_records)})
print({'label_distribution': val_manifest['label_distribution']})
print('Sample record:', json.dumps(train_records[0], indent=2))

## 5. Evaluate pre-adaptation zero-shot baseline

Before running adaptation, we benchmark the zero-shot baseline accuracy of the pretrained checkpoint on our held-out validation set. `pipe.evaluate(val_records, labels=ADAPT_CLASSES)` computes exact-span micro precision, recall, and F1 score, alongside per-class performance breakdowns. This provides an objective baseline against which adaptation gains are quantified.

In [ ]:
baseline_eval = pipe.evaluate(val_records, labels=ADAPT_CLASSES)
print('Pre-adaptation Baseline Metrics (Validation Split):')
print(json.dumps(baseline_eval, indent=2))

## 6. Adapt model via bounded fine-tuning

We now adapt GLiNER to our target domain. Calling `pipe.adapt(train_records, val_records, ...)` executes a lightweight, native PyTorch fine-tuning loop: the mDeBERTa-v3 text encoder backbone is frozen (`freeze_text_encoder=True`), and only the span representation and prompt projection layers are updated using AdamW. This prevents catastrophic forgetting, keeps memory consumption low, and enables fast convergence even on CPU or modest GPU instances.

In [ ]:
import time

EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records=train_records,
    val_records=val_records,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    freeze_text_encoder=True,
    seed=SEED,
)
elapsed = time.perf_counter() - started

print(f'Adaptation completed in {elapsed:.2f}s.')
print('Training History:')
for step in adapt_result['history']:
    print(f"  Epoch {step['epoch']}: Loss={step['loss']:.4f}, Val F1={step.get('val_f1', 0.0):.4f}")

## 7. Evaluate post-adaptation metrics and deltas

We evaluate the adapted model on the validation split and calculate the exact performance delta compared to the pre-adaptation zero-shot baseline. The evaluation report records micro/macro precision, recall, and F1 improvements, as well as per-class metrics, and saves the result to `outputs/gliner_ner_evaluation_report.json`.

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)
adapted_eval = pipe.evaluate(val_records, labels=ADAPT_CLASSES)

delta_f1 = round(adapted_eval['micro']['f1'] - baseline_eval['micro']['f1'], 4)
delta_precision = round(adapted_eval['micro']['precision'] - baseline_eval['micro']['precision'], 4)
delta_recall = round(adapted_eval['micro']['recall'] - baseline_eval['micro']['recall'], 4)

eval_report = {
    'task': 'domain-adapted named-entity recognition',
    'domain': 'biomedical / clinical',
    'labels': list(ADAPT_CLASSES),
    'baseline_eval': baseline_eval,
    'adapted_eval': adapted_eval,
    'delta': {
        'f1_delta': delta_f1,
        'precision_delta': delta_precision,
        'recall_delta': delta_recall,
    },
    'training_summary': adapt_result,
}

with open('outputs/gliner_ner_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(eval_report, f, indent=2)

print('Post-adaptation Validation Metrics:')
print(json.dumps(adapted_eval, indent=2))
print(f"\nMicro F1 Delta: {delta_f1:+.4f} (Baseline: {baseline_eval['micro']['f1']:.4f} -> Adapted: {adapted_eval['micro']['f1']:.4f})")

## 8. Run inference on unseen domain text

Now we test the adapted pipeline on an unseen clinical sentence. `pipe.detect` extracts domain entities with character spans and confidence scores. The output is validated to guarantee that offsets are within text bounds, span texts match substring slices, and all assigned labels exist within the requested vocabulary.

In [ ]:
test_sentence = 'Pembrolizumab blocks PDCD1 receptor to treat metastatic melanoma in adult patients.'
inference_result = pipe.detect(test_sentence, labels=ADAPT_CLASSES, threshold=DEFAULT_THRESHOLD)

entities = inference_result['entities']
checks = {
    'offsets_inside_text': all(0 <= e['start'] < e['end'] <= len(test_sentence) for e in entities),
    'span_text_matches': all(test_sentence[e['start']:e['end']] == e['text'] for e in entities),
    'labels_valid': all(e['label'] in ADAPT_CLASSES for e in entities),
}
assert all(checks.values()), f'Inference failed sanity check: {checks}'

print(f'Input: "{test_sentence}"')
print(f'Detected {len(entities)} entities:')
for i, e in enumerate(entities, 1):
    print(f"  {i}. [{e['start']:>2}:{e['end']:>2}] {e['label']:<14} (score={e['score']:.4f}): {e['text']}")

## 9. Export adapter artifact and verify reload parity

To deploy the adapted model without duplicating the 1.16 GB base checkpoint, `pipe.save_artifact` exports only the trained adapter weights and metadata into a portable `.pt` file (`outputs/gliner_ner_adapter.pt`). We then instantiate a fresh pipeline from the base weights via `GLiNERPipeline.from_artifact` and verify that predictions on unseen text match the in-memory adapted model with complete bit-level parity.

In [ ]:
artifact_path = Path('outputs/gliner_ner_adapter.pt')
pipe.save_artifact(
    artifact_path,
    metadata={
        'domain': 'biomedical',
        'classes': list(ADAPT_CLASSES),
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'metrics': adapted_eval['micro'],
    },
)
print(f'Saved adapter artifact: {artifact_path} ({artifact_path.stat().st_size / (1024*1024):.2f} MB)')

# Verify fresh reload
reloaded_pipe = GLiNERPipeline.from_artifact(
    artifact_path,
    weights_dir=WEIGHTS_DIR,
    encoder_dir=ENCODER_WEIGHTS_DIR,
)
reloaded_res = reloaded_pipe.detect(test_sentence, labels=ADAPT_CLASSES, threshold=DEFAULT_THRESHOLD)
reloaded_entities = reloaded_res['entities']

# Verify exact prediction parity
assert len(entities) == len(reloaded_entities), 'Parity failure: entity count mismatch'
for e_orig, e_rel in zip(entities, reloaded_entities, strict=True):
    assert e_orig['start'] == e_rel['start'] and e_orig['end'] == e_rel['end'], 'Offset parity failure'
    assert e_orig['label'] == e_rel['label'], 'Label parity failure'
    assert abs(e_orig['score'] - e_rel['score']) < 1e-5, 'Score parity failure'
print('Parity Verification Passed: Reloaded artifact matches adapted pipeline identically.')

## 10. Export outputs and lineage manifest

We finalize the tutorial by writing two standard output files under `outputs/`: `gliner_ner_result.json` containing complete provenance, configuration parameters, baseline and adapted evaluation metrics, and runtime specifications; and `gliner_ner_entities.csv` containing tabular entity spans.

In [ ]:
import csv
import platform

result_payload = {
    'task': 'named-entity recognition domain adaptation',
    'pipeline_class': 'GLiNERPipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'encoder_model_id': ENCODER_MODEL_ID,
    'encoder_revision': ENCODER_REVISION,
    'encoder_license': ENCODER_LICENSE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'notebook_source': NOTEBOOK_SOURCE,
    'evaluation_report': eval_report,
    'inference_sample': {
        'text': test_sentence,
        'entities': entities,
    },
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'gliner': gliner.__version__,
        'device': pipe.device,
    },
}

with open('outputs/gliner_ner_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

with open('outputs/gliner_ner_entities.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['index', 'start', 'end', 'label', 'score', 'text'])
    for idx, e in enumerate(entities, 1):
        writer.writerow([idx, e['start'], e['end'], e['label'], f"{e['score']:.6f}", e['text']])

print('Generated release artifacts under outputs/:')
for fname in sorted(os.listdir('outputs')):
    fpath = Path('outputs') / fname
    print(f'  - {fname} ({fpath.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

Domain adaptation tunes span representation projections to align candidate text representations with custom entity types. Freezing the mDeBERTa backbone ensures the model retains its linguistic representation power while preventing catastrophic forgetting on small datasets. Each returned span represents an exact character slice in the original text; the score reflects the model's uncalibrated similarity between span representation and label embedding. Exact-span F1 requires exact agreement on `(start, end, label)` triples: boundary mismatches of even one character receive zero credit. The exported adapter artifact carries only trainable parameters and lineage headers, enabling lightweight distribution and reproducibility.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate metrics against pre-adaptation baseline, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or generalized performance across unseen domains.

## References

- Repository README: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/gliner-ner-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/urchade/gliner_multi-v2.1
- Encoder assets: https://huggingface.co/microsoft/mdeberta-v3-base
- Upstream code: https://github.com/urchade/GLiNER
- GLiNER paper: https://arxiv.org/abs/2311.08526